In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Practice").getOrCreate()
spark

In [7]:
import kagglehub
import os

# 1. Télécharger ou récupérer le dataset depuis le cache Kaggle sur Colab
path = kagglehub.dataset_download("uciml/mushroom-classification")
print("Chemin du dossier :", path)

# 2. Le fichier CSV s'y trouve généralement sous le nom "mushrooms.csv"
csv_path = os.path.join(path, "mushrooms.csv")

# 3. Charger le fichier avec Spark
df_mushrooms = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_path)

# 4. Afficher les premières lignes pour vérifier
df_mushrooms.show(5)

Using Colab cache for faster access to the 'mushroom-classification' dataset.
Chemin du dossier : /kaggle/input/mushroom-classification
+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+
|class|cap-shape|cap-surface|cap-color|bruises|odor|gill-attachment|gill-spacing|gill-size|gill-color|stalk-shape|stalk-root|stalk-surface-above-ring|stalk-surface-below-ring|stalk-color-above-ring|stalk-color-below-ring|veil-type|veil-color|ring-number|ring-type|spore-print-color|population|habitat|
+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+------

In [10]:
from google.colab import files
uploaded = files.upload()

Saving car_data.csv to car_data.csv


In [11]:
df_pyspark = spark.read.csv("car_data.csv",inferSchema=True, header=True)

In [12]:
df_pyspark.show(5)

+------+------------+-----+-------+--------+------+--------+
|buying|maintainence|doors|persons|lug_boot|safety|car_type|
+------+------------+-----+-------+--------+------+--------+
| vhigh|       vhigh|    2|      2|   small|   low|   unacc|
| vhigh|       vhigh|    2|      2|   small|   med|   unacc|
| vhigh|       vhigh|    2|      2|   small|  high|   unacc|
| vhigh|       vhigh|    2|      2|     med|   low|   unacc|
| vhigh|       vhigh|    2|      2|     med|   med|   unacc|
+------+------------+-----+-------+--------+------+--------+
only showing top 5 rows


In [14]:
df_pyspark.printSchema()

root
 |-- buying: string (nullable = true)
 |-- maintainence: string (nullable = true)
 |-- doors: string (nullable = true)
 |-- persons: string (nullable = true)
 |-- lug_boot: string (nullable = true)
 |-- safety: string (nullable = true)
 |-- car_type: string (nullable = true)



In [15]:
from pyspark.ml.feature import StringIndexer

In [17]:
categoricalColumns = ["buying","maintainence","doors","persons","lug_boot","safety","car_type"]

In [18]:
l = []

In [21]:
for categoricalCol in categoricalColumns:
      stringIndexer = StringIndexer(inputCol = categoricalCol, outputCol = categoricalCol+"_encoded").fit(df_pyspark)

In [22]:
df_pyspark = stringIndexer.transform(df_pyspark)

In [23]:
df_pyspark = df_pyspark.withColumn(categoricalCol+"_encoded", df_pyspark[categoricalCol+"_encoded"].cast('int'))

In [27]:
encoded_df = df_pyspark.select(
    "buying_encoded",
    "doors_encoded",  # <-- C'est ici qu'il manquait "_encoded" !
    "maintainence_encoded",
    "persons_encoded",
    "lug_boot_encoded",
    "safety_encoded",
    "car_type_encoded"
)

In [28]:
encoded_df.show(5)

+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+
|buying_encoded|doors_encoded|maintainence_encoded|persons_encoded|lug_boot_encoded|safety_encoded|car_type_encoded|
+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+
|           3.0|          0.0|                 3.0|            0.0|             2.0|           1.0|               0|
|           3.0|          0.0|                 3.0|            0.0|             2.0|           2.0|               0|
|           3.0|          0.0|                 3.0|            0.0|             2.0|           0.0|               0|
|           3.0|          0.0|                 3.0|            0.0|             1.0|           1.0|               0|
|           3.0|          0.0|                 3.0|            0.0|             1.0|           2.0|               0|
+--------------+-------------+--------------------+-------------

In [29]:
from pyspark.ml.feature import VectorAssembler

In [30]:
featureAssembler = VectorAssembler(inputCols=["buying_encoded","doors_encoded","maintainence_encoded","persons_encoded","lug_boot_encoded","safety_encoded"],outputCol="features")

In [31]:
output = featureAssembler.transform(encoded_df)

In [32]:
output.select("features","car_type_encoded").show(5)

+--------------------+----------------+
|            features|car_type_encoded|
+--------------------+----------------+
|[3.0,0.0,3.0,0.0,...|               0|
|[3.0,0.0,3.0,0.0,...|               0|
|[3.0,0.0,3.0,0.0,...|               0|
|[3.0,0.0,3.0,0.0,...|               0|
|[3.0,0.0,3.0,0.0,...|               0|
+--------------------+----------------+
only showing top 5 rows


In [33]:
train, test = output.randomSplit([0.8, 0.2], seed=17)

In [34]:
print("Size of traning data:",train.count())
print("Size of testing data:",test.count())

Size of traning data: 1377
Size of testing data: 351


In [35]:
from pyspark.ml.classification import LogisticRegression

In [36]:
lr = LogisticRegression(featuresCol = 'features', labelCol = 'car_type_encoded', maxIter=10)

In [37]:
lrModel=lr.fit(train)

In [38]:
predictions = lrModel.transform(test)

In [39]:
predictions.show(5)

+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+-------------------+--------------------+--------------------+----------+
|buying_encoded|doors_encoded|maintainence_encoded|persons_encoded|lug_boot_encoded|safety_encoded|car_type_encoded|           features|       rawPrediction|         probability|prediction|
+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+-------------------+--------------------+--------------------+----------+
|           0.0|          0.0|                 0.0|            0.0|             0.0|           0.0|               0|          (6,[],[])|[1.99198136762973...|[0.76125517518711...|       0.0|
|           0.0|          0.0|                 0.0|            0.0|             2.0|           0.0|               0|      (6,[4],[2.0])|[3.26816128608158...|[0.90473344519298...|       0.0|
|           0.0|          0.0|                 0.0

In [40]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [41]:
evaluator = MulticlassClassificationEvaluator()

In [42]:
evaluator.setLabelCol("car_type_encoded")

MulticlassClassificationEvaluator_7b60e41648e3

In [43]:
evaluator.setPredictionCol("prediction")

MulticlassClassificationEvaluator_7b60e41648e3

In [44]:
evaluator.evaluate(predictions)

0.5901985607326307

In [49]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator()
evaluator.setLabelCol("car_type_encoded")
evaluator.setPredictionCol("prediction")
print("Test Area Under ROC: ",evaluator.evaluate(predictions))

Test Area Under ROC:  0.5901985607326307


In [50]:
encoded_df.show(5)

+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+
|buying_encoded|doors_encoded|maintainence_encoded|persons_encoded|lug_boot_encoded|safety_encoded|car_type_encoded|
+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+
|           3.0|          0.0|                 3.0|            0.0|             2.0|           1.0|               0|
|           3.0|          0.0|                 3.0|            0.0|             2.0|           2.0|               0|
|           3.0|          0.0|                 3.0|            0.0|             2.0|           0.0|               0|
|           3.0|          0.0|                 3.0|            0.0|             1.0|           1.0|               0|
|           3.0|          0.0|                 3.0|            0.0|             1.0|           2.0|               0|
+--------------+-------------+--------------------+-------------

In [51]:
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


In [52]:
#Training Model

In [53]:
dt = DecisionTreeClassifier(featuresCol = 'features', labelCol = 'car_type_encoded', maxDepth = 3)
dtModel = dt.fit(train)

In [54]:
#Prediction

In [55]:
predictions = dtModel.transform(test)

In [56]:
#Evaluating the performance

In [57]:
evaluator = MulticlassClassificationEvaluator()
evaluator.setLabelCol("car_type_encoded")
evaluator.setPredictionCol("prediction")

MulticlassClassificationEvaluator_6ddd9cd2fa9c

In [59]:
print("Test Area Under ROC: ",evaluator.evaluate(predictions))

Test Area Under ROC:  0.7161258193778519


In [61]:
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Training Model
dt = DecisionTreeClassifier(featuresCol = 'features', labelCol = 'car_type_encoded', maxDepth = 3)
dtModel = dt.fit(train)

#Prediction
predictions = dtModel.transform(test)

#Evaluating the performance
evaluator = MulticlassClassificationEvaluator()
evaluator.setLabelCol("car_type_encoded")
evaluator.setPredictionCol("prediction")
print("Test Area Under ROC: ",evaluator.evaluate(predictions))



Test Area Under ROC:  0.7161258193778519


In [62]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [63]:
#Modèle d'entraînement
rf = RandomForestClassifier(featuresCol = 'features', labelCol = 'car_type_encoded', numTrees = 500, maxDepth = 10)
rfModel = rf.fit(train)

In [64]:
#Prédictions
predictions = rfModel.transform(test)

In [65]:
#Évaluation des performances
evaluator = MulticlassClassificationEvaluator()
evaluator.setLabelCol("car_type_encoded")
evaluator.setPredictionCol("prediction")
print("Aire sous la courbe ROC : ",evaluator.evaluate(predictions))

Aire sous la courbe ROC :  0.9717189494381577


In [66]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Training Model
rf = RandomForestClassifier(featuresCol = 'features', labelCol = 'car_type_encoded', numTrees = 500, maxDepth = 10)
rfModel = rf.fit(train)

#Prediction
predictions = rfModel.transform(test)

#Evaluating the performance
evaluator = MulticlassClassificationEvaluator()
evaluator.setLabelCol("car_type_encoded")
evaluator.setPredictionCol("prediction")
print("Test Area Under ROC: ",evaluator.evaluate(predictions))

Test Area Under ROC:  0.9717189494381577
